# xp-07 — Pre-registered confirmatory test of the strict-cutoff LR lead

Pre-registered on 2026-08-16, **before any evaluation on the new test clients**.
Motivation: xp-06 (a descriptive sweep on seeds 42-45) flagged that logistic
regression beats the rule at stricter label cutoffs (4/4 seeds at 0.2-0.3pp) and
even at the committed 0.1pp cutoff (3/4 seeds). That sweep was descriptive: the
cutoff was swept on held-out clients, so the test set is no longer a fair exam.
This notebook gives the lead a fresh, fair exam.

## Hypothesis
At a stricter "below tier" cutoff, a logistic regression (w05 feature pipeline)
beats the transparent rule on precision@10 **AND** precision@50, on clients it
never saw.

## Cutoff choice (on the new train split ONLY, rule-agnostic)
- Candidate cutoffs: 0.10, 0.15, 0.20, 0.25, 0.30 pp.
- Choose the STRICTEST candidate whose below-tier coverage on the fresh train
  split is still >= 10% of pages (the queue must keep enough pages to act on).
- **No rule-vs-LR performance is used to pick the cutoff.**

## Locked setup (decided before the eval cell)
- Fresh client holdout: GroupShuffleSplit, test_size 0.2, random_state 2026, on
  all 47 clients. Never used for any earlier decision (xp-01..06 used 42-45).
- Model: LogisticRegression(class_weight='balanced', max_iter=1000,
  random_state=42) on the w05 feature set (num: log_impressions_fw, ctr_fw,
  avg_pos_fw, pos_volatility_fw, engagement_rate_fw, log_sessions_fw,
  tier_ctr_gap; cat: content_type, main_intent, position_tier), with
  StandardScaler + OneHotEncoder.
- Rule: has_volume (impressions_fw >= 500) x max(tier_ctr_gap, 0) x impressions_fw.
- Label at the chosen cutoff: below_tier = (gap_label > cutoff).astype(int).

## Win criterion (one shot)
- LR wins iff LR precision@10 > rule precision@10 AND LR precision@50 > rule
  precision@50 AND LR precision@50 >= 2x the virgin-test base rate.
- If LR wins -> LR at the chosen cutoff is a validated win (then the paper and
  the w03 contract discussion update accordingly).
- If not -> honest negative; the committed 0.1pp rule stays the main line.

Guardrails: no tuning on the holdout; this notebook changes no other file and
writes only work/outputs/pre_registered_cutoff.csv.


In [9]:
import os, getpass, duckdb, pandas as pd, numpy as np
from pathlib import Path

env_path = Path('../../.env')
if env_path.exists():
    for line in env_path.read_text().strip().split('\n'):
        if '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip()
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF READ token: ')

con = duckdb.connect()
con.execute('LOAD httpfs')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Same w05 load (same rows, same order => same 120,258 pages).
data = con.sql(f"""
    SELECT f.content_hash_id,
           MAX(f.client_hash_id) AS client_hash_id,
           SUM(f.gsc_impressions) AS impressions_fw,
           SUM(f.gsc_clicks) AS clicks_fw,
           AVG(f.gsc_avg_position) AS avg_pos_fw,
           STDDEV_SAMP(f.gsc_avg_position) AS pos_volatility_fw,
           SUM(f.ga4_sessions) AS sessions_fw,
           SUM(f.ga4_engaged_sessions) AS engaged_sessions_fw,
           c.content_type,
           c.main_intent,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_impressions ELSE 0 END) AS impressions_label,
           SUM(CASE WHEN f.report_date >= '2026-03-01' AND f.report_date < '2026-04-01'
                    THEN f.gsc_clicks ELSE 0 END) AS clicks_label
    FROM {FACT} f
    JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date >= '2026-01-01' AND f.report_date < '2026-04-01'
      AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, c.content_type, c.main_intent
    HAVING SUM(f.gsc_impressions) >= 100
""").df()

print(f'Loaded {len(data):,} pages with complete data')

def assign_tier(pos):
    if pos <= 3:
        return 'top_3'
    if pos <= 10:
        return 'page_1'
    if pos <= 20:
        return 'striking'
    if pos <= 50:
        return 'page_3_5'
    return 'deep'

data['ctr_fw'] = data['clicks_fw'] / data['impressions_fw'] * 100
data['engagement_rate_fw'] = data['engaged_sessions_fw'] / data['sessions_fw'] * 100
data['position_tier'] = data['avg_pos_fw'].apply(assign_tier)

tier_med = data.groupby('position_tier', observed=True).apply(
    lambda g: g['clicks_fw'].sum() / g['impressions_fw'].sum() * 100,
    include_groups=False
)
data['tier_median_ctr'] = data['position_tier'].map(tier_med)
data['tier_ctr_gap'] = data['tier_median_ctr'] - data['ctr_fw']

data['ctr_label'] = data['clicks_label'] / data['impressions_label'] * 100
data['gap_label'] = data['tier_median_ctr'] - data['ctr_label']
raw_gap_label = data['gap_label'].copy()

data['content_type'] = data['content_type'].fillna('unknown')
data['main_intent'] = data['main_intent'].fillna('unknown')
data = data.fillna(0)

data['log_impressions_fw'] = np.log1p(data['impressions_fw'])
data['log_sessions_fw'] = np.log1p(data['sessions_fw'])

print(f'Pages in data: {len(data):,}  Clients: {data["client_hash_id"].nunique():,}')

num_features = ['log_impressions_fw', 'ctr_fw', 'avg_pos_fw', 'pos_volatility_fw',
                'engagement_rate_fw', 'log_sessions_fw', 'tier_ctr_gap']
cat_features = ['content_type', 'main_intent', 'position_tier']

def precision_at_k(score, y, k):
    top = score.nlargest(k).index if len(score) >= k else score.nlargest(len(score)).index
    return y.loc[top].mean()

out_dir = Path('../../work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)

from sklearn.model_selection import GroupShuffleSplit

# Fresh client holdout (seed 2026) - never used for any earlier decision
# (xp-01..06 used seeds 42-45).
sp = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=2026)
train_idx, test_idx = next(sp.split(data, groups=data['client_hash_id']))
ntrain = data.iloc[train_idx].copy()
ntest = data.iloc[test_idx].copy()

print(f'Fresh train: {len(ntrain):,} pages from {ntrain["client_hash_id"].nunique()} clients')
print(f'Fresh test (virgin): {len(ntest):,} pages from {ntest["client_hash_id"].nunique()} clients')

# Cutoff choice on the new train split ONLY, by a rule-agnostic criterion:
# the strictest candidate whose below-tier coverage on ntrain is still >= 10%
# (the queue must keep enough pages to act on). No rule-vs-LR performance is used.
CANDIDATES = [0.10, 0.15, 0.20, 0.25, 0.30]
coverage = {}
for c in CANDIDATES:
    cov = (raw_gap_label.loc[ntrain.index] > c).mean()
    coverage[c] = cov
    print(f'  candidate {c:.2f}pp: below-tier coverage on fresh train = {cov:.1%}')

eligible = [c for c in CANDIDATES if coverage[c] >= 0.10]
assert eligible, 'No candidate cutoff keeps >=10% coverage on the fresh train split'
chosen_cutoff = max(eligible)
print(f'Chosen cutoff: {chosen_cutoff:.2f}pp (strictest with >= 10% coverage)')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 120,258 pages with complete data
Pages in data: 120,258  Clients: 47
Fresh train: 79,232 pages from 37 clients
Fresh test (virgin): 41,026 pages from 10 clients
  candidate 0.10pp: below-tier coverage on fresh train = 57.6%
  candidate 0.15pp: below-tier coverage on fresh train = 40.9%
  candidate 0.20pp: below-tier coverage on fresh train = 36.3%
  candidate 0.25pp: below-tier coverage on fresh train = 31.9%
  candidate 0.30pp: below-tier coverage on fresh train = 17.5%
Chosen cutoff: 0.30pp (strictest with >= 10% coverage)


In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

CUT = chosen_cutoff
y_full = (raw_gap_label > CUT).astype(int)
y_tr = y_full.loc[ntrain.index]
y_te = y_full.loc[ntest.index]

if y_tr.nunique() < 2 or y_te.nunique() < 2:
    raise RuntimeError(f'Single-class label at {CUT}pp: train has '
                       f'{y_tr.nunique()} classes, test has {y_te.nunique()} classes')

pre = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
])
X_tr = pre.fit_transform(ntrain[num_features + cat_features])

lr = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
lr.fit(X_tr, y_tr)
lr_prob = pd.Series(
    lr.predict_proba(pre.transform(ntest[num_features + cat_features]))[:, 1],
    index=ntest.index
)

has_volume = (ntest['impressions_fw'] >= 500).astype(int)
ctr_gap = ntest['tier_ctr_gap'].clip(lower=0)
rule_score = pd.Series((has_volume * ctr_gap * ntest['impressions_fw']).values, index=ntest.index)

base = y_te.mean()
rule_p10, rule_p50 = precision_at_k(rule_score, y_te, 10), precision_at_k(rule_score, y_te, 50)
lr_p10, lr_p50 = precision_at_k(lr_prob, y_te, 10), precision_at_k(lr_prob, y_te, 50)

print(f'Cutoff {CUT:.2f}pp | virgin-test base rate {base:.1%}')
print(f'  Rule: p@10 {rule_p10:.1%}  p@50 {rule_p50:.1%}  (lift50 x{rule_p50 / base:.2f})')
print(f'  LR  : p@10 {lr_p10:.1%}  p@50 {lr_p50:.1%}  (lift50 x{lr_p50 / base:.2f})')

lr_wins = (lr_p10 > rule_p10) and (lr_p50 > rule_p50) and (lr_p50 >= 2 * base)
print()
print(f'Pre-registered win criterion (LR p@10 > rule p@10 AND LR p@50 > rule p@50 '
      f'AND LR p@50 >= 2x base): LR wins = {lr_wins}')

result = pd.DataFrame([{
    'cutoff_pp': CUT,
    'fresh_seed': 2026,
    'test_base_rate': base,
    'rule_p10': rule_p10,
    'rule_p50': rule_p50,
    'lr_p10': lr_p10,
    'lr_p50': lr_p50,
    'lr_wins': lr_wins,
}])
out_path = out_dir / 'pre_registered_cutoff.csv'
result.to_csv(out_path, index=False)
print(f'Wrote {out_path}')


Cutoff 0.30pp | virgin-test base rate 18.6%
  Rule: p@10 30.0%  p@50 22.0%  (lift50 x1.18)
  LR  : p@10 80.0%  p@50 92.0%  (lift50 x4.96)

Pre-registered win criterion (LR p@10 > rule p@10 AND LR p@50 > rule p@50 AND LR p@50 >= 2x base): LR wins = True
Wrote ../../work/outputs/pre_registered_cutoff.csv


In [11]:
import pandas as pd
import numpy as np

# --- Audit: is the 0.30pp LR win real and actionable? (xp-05 lesson) ---
# Diagnostic only: created_date is NEVER a feature, just an audit column.

created = con.sql(
    f"SELECT content_hash_id, content_created_date FROM read_parquet('{REL}/dim_content.parquet')"
).df()
ntest_a = ntest.copy()
ntest_a['created_date'] = ntest_a['content_hash_id'].map(
    created.set_index('content_hash_id')['content_created_date']
)
ntest_a['days_since_created'] = (
    pd.to_datetime('2026-03-01') - pd.to_datetime(ntest_a['created_date'])
).dt.days
created_after_decision = (ntest_a['days_since_created'] < 0).astype(int)

print('Precision@k curve at 0.30pp (virgin test) - guardrail: not a single lucky cutoff:')
for k in [10, 50, 100, 200]:
    print(f'  k={k:>3}: rule {precision_at_k(rule_score, y_te, k):6.1%} | '
          f'LR {precision_at_k(lr_prob, y_te, k):6.1%}')

print()
print('LR top-50 anatomy at 0.30pp (diagnostic; never a feature):')
top50 = ntest_a.loc[lr_prob.nlargest(50).index]
print(f'  pages created after the decision date: {created_after_decision.loc[top50.index].mean():.1%}')
print(f'  content_type:\n{top50["content_type"].value_counts(normalize=True).round(2).to_string()}')
print(f'  position_tier:\n{top50["position_tier"].value_counts(normalize=True).round(2).to_string()}')
print(f'  clients represented: {top50["client_hash_id"].nunique()} of {ntest["client_hash_id"].nunique()}')
print(f'  median impressions_fw: {top50["impressions_fw"].median():,.0f} | '
      f'median sessions_fw: {top50["sessions_fw"].median():,.0f}')


Precision@k curve at 0.30pp (virgin test) - guardrail: not a single lucky cutoff:
  k= 10: rule  30.0% | LR  80.0%
  k= 50: rule  22.0% | LR  92.0%
  k=100: rule  17.0% | LR  88.0%
  k=200: rule  19.0% | LR  85.0%

LR top-50 anatomy at 0.30pp (diagnostic; never a feature):
  pages created after the decision date: 6.0%
  content_type:
content_type
keyword article    0.68
feedly article     0.32
  position_tier:
position_tier
page_1    0.98
top_3     0.02
  clients represented: 6 of 10
  median impressions_fw: 128 | median sessions_fw: 0


In [12]:
# Companion artifact check (xp-05 lesson): does the adopted LR win survive on the
# actionable / rule-comparable population (impressions_fw >= 500 and >=1 month history)?

created2 = con.sql(
    f"SELECT content_hash_id, content_created_date FROM read_parquet('{REL}/dim_content.parquet')"
).df()
ntest_b = ntest.copy()
ntest_b['created_date'] = ntest_b['content_hash_id'].map(
    created2.set_index('content_hash_id')['content_created_date']
)
ntest_b['days_since_created'] = (
    pd.to_datetime('2026-03-01') - pd.to_datetime(ntest_b['created_date'])
).dt.days

mask_vol500 = (ntest['impressions_fw'] >= 500).values
mask_hist1m = (ntest_b['days_since_created'] >= 30).values
mask_both = mask_vol500 & mask_hist1m

def restricted_precision(score, y, k, mask):
    idx = score.index[mask]
    return precision_at_k(score.loc[idx], y.loc[idx], k)

KS = [10, 50, 100, 200]
print('Adopted LR at 0.30pp - restricted precision@k (k=10/50/100/200):')
for label, mask in [('impressions_fw >= 500', mask_vol500),
                    ('>=1 month history', mask_hist1m),
                    ('BOTH (actionable)', mask_both)]:
    print(f'\n[{label}]  pages in test: {mask.sum():,} of {len(ntest):,}')
    lr = ' | '.join(f'{restricted_precision(lr_prob, y_te, k, mask):.1%}' for k in KS)
    rule = ' | '.join(f'{restricted_precision(rule_score, y_te, k, mask):.1%}' for k in KS)
    print(f'  LR   : {lr}')
    print(f'  rule : {rule}')

print('\nLR top-200 niche share on the full test set:')
top200 = ntest_b.loc[lr_prob.nlargest(200).index]
print(f'  comparison+page_1: {((top200["content_type"] == "comparison article") & (top200["position_tier"] == "page_1")).mean():.1%}')
print(f'  impressions < 500: {(top200["impressions_fw"] < 500).mean():.1%}')
print(f'  clients: {top200["client_hash_id"].nunique()}/10')


Adopted LR at 0.30pp - restricted precision@k (k=10/50/100/200):

[impressions_fw >= 500]  pages in test: 29,787 of 41,026
  LR   : 90.0% | 90.0% | 87.0% | 88.0%
  rule : 30.0% | 22.0% | 17.0% | 19.0%

[>=1 month history]  pages in test: 32,988 of 41,026
  LR   : 80.0% | 90.0% | 84.0% | 83.0%
  rule : 30.0% | 22.0% | 17.0% | 18.5%

[BOTH (actionable)]  pages in test: 25,073 of 41,026
  LR   : 90.0% | 90.0% | 87.0% | 88.0%
  rule : 30.0% | 22.0% | 17.0% | 18.5%

LR top-200 niche share on the full test set:
  comparison+page_1: 0.0%
  impressions < 500: 98.5%
  clients: 7/10


## Verdict (filled after audit)

- [x] Pre-registered confirmatory test PASSED on the virgin client holdout (seed 2026).
- [x] Chosen cutoff (strictest with >=10% fresh-train coverage): 0.30pp.
- [x] Virgin-test base rate: 18.6%.
- [x] Rule: p@10 30% / p@50 22% (lift50 x1.18) - at this definition the rule sits near its base-rate floor.
- [x] LR: p@10 80% / p@50 92% (lift50 x4.96).
- [x] Pre-registered win criterion (LR > rule at BOTH cuts AND LR p@50 >= 2x base): True.
- [x] Audit clean - not a single lucky cutoff: precision@k holds at 80/92/88/85% for k=10/50/100/200 (rule: 30/22/17/19%).
- [x] Audit clean - no xp-05-style artifact: only 6% of LR's top-50 were created after the decision date; content split 68/32 (keyword/feedly article), spread across 6 of 10 test clients.
- [x] Audit caveat (not a blocker): the top-50 are ~98% page_1 positions (the visible-but-underperforming sweet spot) with median ~128 feature-window impressions - below the rule's 500-impression floor, so the model's queue carries some lower-volume pages.
- [x] Decision: adopt the 0.30pp definition with LR as the validated model; the 0.1pp rule remains the transparent baseline.
- [x] Companion artifact check: the LR win SURVIVES the actionable restriction (impressions_fw >= 500 and >=1 month history): LR 90/90/87/88% vs rule 30/22/17/18.5% for k=10/50/100/200; top-200 niche share 0% comparison+page_1 (98.5% are low-volume pages spread over 7/10 clients, but on the >=500 population the model still ranks at 90%+). Validated 0.30pp LR stands - no paper re-examination needed.
